## Instrução de Refatoração e Automação: Scraper TJSP Capa

O objetivo desta tarefa é modernizar o scraper do TJSP para utilizar `curl-cffi`, garantindo que as requisições simulem um navegador real para evitar bloqueios, e criar um script de coleta que suporte interrupções (capacidade de *resume*).

### 1. Setup e Workflow de Git (Obrigatório)
Diferente de um fluxo comum, **o Pull Request (PR) deve ser aberto antes mesmo da primeira linha de código escrita**. Isso permite que o acompanhamento e a mentoria aconteçam em tempo real.

1.  **Criação da Branch:**
    ```bash
    git checkout -b feature/refactor-tjsp-curl-cffi
    ```
2.  **Abertura do Draft PR:**
    * Faça um commit vazio: `git commit --allow-empty -m "Iniciando refatoração e automação do scraper"`
    * Dê o push e, no GitHub, abra um **Draft Pull Request** apontando para a `main`.

---

### 2. Refatoração da Classe (`src/scrapers/`)
A classe em `src/scrapers/scraper_tjsp_capa_request.py` deve ser atualizada para usar o `curl-cffi`.

* **Instalação:** `uv add curl-cffi beautifulsoup4 pandas`
* **Mudança Principal:** Substituir `requests.Session()` por `curl_cffi.requests.Session(impersonate="chrome120")`.
* **Vantagem:** O `curl-cffi` mimetiza o "aperto de mão" (TLS Fingerprint) de um navegador real, algo que o `requests` comum não faz, sendo facilmente bloqueado pelo e-SAJ.
* **Checagem Técnica:** Checagem técnica dos dicionários padrões para identificação de elementos HTML com especialista do Direito.

---

### 3. Script de Coleta Persistente (`scripts/`)
Após validar a classe, você deve criar o script `scripts/coleta_PGE_GPDR_tjsp_fase_capa.py`. O foco aqui é a **resiliência**. Se a coleta parar (queda de luz, erro de rede), ela deve recomeçar de onde parou.

#### Lógica de "Resume":
O script deve usar o **número do processo** como chave primária.
1. Carregar a lista total de processos do arquivo `data/PGE.GPDR.json`.
2. Verificar no arquivo de saída (`data/coleta_tjsp_resultados.csv`) quais números já foram processados.
3. Filtrar a lista e processar apenas os itens restantes.

#### Estrutura do Código Sugerida:
```python
import json
import os
import pandas as pd
from src.scrapers.scraper_tjsp_capa_request import TJSPScraper 
import time

INPUT_FILE = "data/PGE.GPDR.json"
OUTPUT_FILE = "data/coleta_tjsp_resultados.json"

def carregar_ja_coletados():
    if not os.path.exists(OUTPUT_FILE):
        return set()
    # Carrega apenas a coluna de números para otimizar memória
    df_existente = pd.read_csv(OUTPUT_FILE, usecols=["F2_Processo"])
    return set(df_existente["F2_Processo"].astype(str).tolist())

def executar_coleta():
    scraper = TJSPScraper()
    with open(INPUT_FILE, 'r') as f:
        dados_base = json.load(f)
    
    ja_feitos = carregar_ja_coletados()
    # Filtra usando o campo 'numero_processo' do JSON
    a_processar = [p for p in dados_base if p['numero_processo'] not in ja_feitos]

    for item in a_processar:
        num = item['numero_processo']
        url = f"https://esaj.tjsp.jus.br/cpopg/show.do?processo.codigo={num}"
        
        try:
            html = scraper.get_html(url)
            resultado = scraper.parse_process(html)
            
            if resultado:
                df_linha = pd.DataFrame([resultado])
                # mode='a' (append) adiciona ao final do arquivo sem sobrescrever
                df_linha.to_json(OUTPUT_FILE, mode='a', index=False, header=not os.path.exists(OUTPUT_FILE))
            
            time.sleep(2) # Delay educado
            
        except Exception as e:
            print(f"Erro no processo {num}: {e}")
            continue

if __name__ == "__main__":
    executar_coleta()
```

---

### 4. Checklist de Entrega (Checklist do PR)
O seu Pull Request será avaliado com base nestes pontos:
* [ ] **Conexão:** O `curl-cffi` está retornando o HTML completo (sem Captcha)?
* [ ] **Robustez:** O script de coleta consegue rodar, ser interrompido e retomar sem duplicar dados?
* [ ] **Parsing:** O uso de `soup.select` está capturando os dados corretamente mesmo em variações de layout?
* [ ] **Segurança:** O arquivo `data/coleta_tjsp_resultados.csv` **não** foi commitado (está no `.gitignore`)?

### 5. Dicas Finais
* **Execução:** Sempre execute o script da raiz do projeto: `uv run python scripts/coleta_tjsp_fase_capa.py`.
* **Dúvidas:** Não tente resolver bloqueios complexos sozinho por muito tempo. Se o e-SAJ começar a retornar `403` ou `429` insistentemente, reporte no PR para ajustarmos o `impersonate` ou o `delay`.

O Draft PR é o seu canal direto de comunicação. Use os comentários para cada decisão técnica!